In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))
from paths import DATA

In [7]:
import requests
import pandas as pd 
import time 



In [8]:
headers = {
    'Origin': 'https://www.premierleague.com',
    'Referer': 'https://www.premierleague.com/',
}

r = requests.get(
    "https://footballapi.pulselive.com/football/competitions/1/compseasons",
    headers=headers
)
print(r.status_code)
print(r.json())
all_fixtures = []
for page in range(38):  
    r = requests.get(
        f"https://footballapi.pulselive.com/football/fixtures?compSeasons=841&page={page}&pageSize=10&sort=asc",
        headers=headers
    )
    data = r.json()
    if not data.get('content'):
        break
    all_fixtures.extend(data['content'])
    print(f"Page {page} — {len(data['content'])} fixtures")

print(f"\n total fixtures: {len(all_fixtures)}")
print(all_fixtures[0])
df_fixtures = pd.json_normalize(all_fixtures)
df_fixtures.to_csv(DATA / 'fixtures_2627.csv', index=False)
print(f"\n total fixtures: {len(df_fixtures)}")
print(df_fixtures.columns.tolist())
print(df_fixtures.head(3))

200
{'pageInfo': {'page': 0, 'numPages': 4, 'pageSize': 10, 'numEntries': 35}, 'content': [{'label': 'English Premier League Season 2026/2027', 'id': 841.0}, {'label': '2025/26', 'id': 777.0}, {'label': '2024/25', 'id': 719.0}, {'label': '2023/24', 'id': 578.0}, {'label': '2022/23', 'id': 489.0}, {'label': '2021/22', 'id': 418.0}, {'label': '2020/21', 'id': 363.0}, {'label': '2019/20', 'id': 274.0}, {'label': '2018/19', 'id': 210.0}, {'label': '2017/18', 'id': 79.0}]}
Page 0 — 10 fixtures
Page 1 — 10 fixtures
Page 2 — 10 fixtures
Page 3 — 10 fixtures
Page 4 — 10 fixtures
Page 5 — 10 fixtures
Page 6 — 10 fixtures
Page 7 — 10 fixtures
Page 8 — 10 fixtures
Page 9 — 10 fixtures
Page 10 — 10 fixtures
Page 11 — 10 fixtures
Page 12 — 10 fixtures
Page 13 — 10 fixtures
Page 14 — 10 fixtures
Page 15 — 10 fixtures
Page 16 — 10 fixtures
Page 17 — 10 fixtures
Page 18 — 10 fixtures
Page 19 — 10 fixtures
Page 20 — 10 fixtures
Page 21 — 10 fixtures
Page 22 — 10 fixtures
Page 23 — 10 fixtures
Page 24 —

In [9]:
print(df_fixtures.columns.tolist())

['teams', 'replay', 'neutralGround', 'status', 'phase', 'outcome', 'attendance', 'fixtureType', 'extraTime', 'shootout', 'goals', 'penaltyShootouts', 'behindClosedDoors', 'id', 'gameweek.id', 'gameweek.compSeason.label', 'gameweek.compSeason.competition.abbreviation', 'gameweek.compSeason.competition.description', 'gameweek.compSeason.competition.level', 'gameweek.compSeason.competition.source', 'gameweek.compSeason.competition.id', 'gameweek.compSeason.id', 'gameweek.gameweek', 'gameweek.competitionPhase.id', 'gameweek.competitionPhase.type', 'gameweek.competitionPhase.gameweekRange', 'kickoff.completeness', 'kickoff.millis', 'kickoff.label', 'kickoff.gmtOffset', 'provisionalKickoff.completeness', 'provisionalKickoff.millis', 'provisionalKickoff.label', 'provisionalKickoff.gmtOffset', 'ground.name', 'ground.city', 'ground.source', 'ground.id', 'clock.secs', 'clock.label']


In [ ]:
df_fixtures['home_team'] = df_fixtures['teams'].apply(lambda x: x[0]['team']['name'] if isinstance(x, list) else None)
df_fixtures['away_team'] = df_fixtures['teams'].apply(lambda x: x[1]['team']['name'] if isinstance(x, list) else None)

df_clean = df_fixtures[[
    'id',
    'gameweek.gameweek',
    'kickoff.label',
    'home_team',
    'away_team',
    'ground.name',
    'ground.city',
    'status'
]].rename(columns={
    'gameweek.gameweek': 'GW',
    'kickoff.label':     'kickoff',
    'ground.name':       'stadium',
    'ground.city':       'city'
})

print(df_clean.head(10).to_string(index=False))
df_clean.to_csv(DATA / 'fixtures_2627.csv', index=False)
print("saved")

      id  GW                    kickoff              home_team              away_team                         stadium        city status
128923.0 1.0 Fri 21 Aug 2026, 20:00 BST                Arsenal          Coventry City                Emirates Stadium      London      C
128926.0 1.0 Sat 22 Aug 2026, 12:30 BST              Hull City      Manchester United                 The MKM Stadium        Hull      C
128925.0 1.0 Sat 22 Aug 2026, 15:00 BST                Everton         Crystal Palace          Hill Dickinson Stadium   Liverpool      C
128927.0 1.0 Sat 22 Aug 2026, 15:00 BST           Ipswich Town             Sunderland                    Portman Road     Ipswich      C
128928.0 1.0 Sat 22 Aug 2026, 15:00 BST      Nottingham Forest           Leeds United                 The City Ground  Nottingham      C
128924.0 1.0 Sat 22 Aug 2026, 17:30 BST              Brentford      Tottenham Hotspur         Gtech Community Stadium   Brentford      C
128929.0 1.0 Sun 23 Aug 2026, 14:00 BST B